### Remove all the rows where Customer Id,Policy ID is null
### Remove all rows where Customer Id not exist in Customer table
### Every policy must have preminum & covergae amount >0
### Validate end_date>start_date

In [0]:
df = spark.sql(" SELECT p.* FROM bronzelayer.policy p INNER JOIN  bronzelayer.customer c  ON p.customer_id = c.customer_id  where p.customer_id is not null and p.policy_id is not null and p.merge_flag = FALSE and p.premium >0 and  p.coverage_amount >0 AND p.end_date > p.start_date ")
display(df)

### Merged the table with merged_date_timestamp as current timesatmp

In [0]:
df.createOrReplaceTempView('clean_policy')
spark.sql("Merge into  silverlayer.policy AS T USING clean_policy AS S ON t.policy_id = s.policy_id WHEN MATCHED THEN UPDATE SET T.policy_type = s.policy_type , T.premium = s.premium, T.end_date = s.end_date, T.start_date = s.start_date, T.coverage_amount =s.coverage_amount, T.customer_id = s.customer_id, T.merge_timestamp = current_timestamp()  WHEN NOT MATCHED THEN INSERT (policy_id, policy_type,premium,end_date, start_date,coverage_amount,customer_id,merge_timestamp ) VALUES  (s.policy_id, s.policy_type,s.premium,s.end_date, s.start_date,s.coverage_amount,s.customer_id,current_timestamp())")

In [0]:
%sql
select * from silverlayer.policy

#Update the merged_flag in the bronze layer

In [0]:
%sql
UPDATE bronzelayer.policy SET merge_flag = True WHERE merge_flag = FALSE